# Quiz 01: Foundations

1. Eight small questions, one at a time.
2. Guess the answer before each run.
3. Then run the cell, then check.

## Words you will see

1. Elastic = a database built for searching.
2. Index = one table of documents.
3. Document = one job posting as JSON.
4. Embedding = numbers that capture a meaning.
5. Semantic search = search by meaning, not words.
6. Aggregation = counting and grouping many documents.
7. RAG = retrieve first, then generate answer.
8. Tool call = model asks Python to run.
9. Notebook = a script cut into slices.

## How to run it

1. Click a code cell, press Shift+Enter.
2. Run the setup cell first, always.

## Setup

1. Run this cell before anything else.
2. It connects to Elastic and Mistral.

In [1]:
import os, json, math
from dotenv import load_dotenv
from mistralai.client import Mistral
from nyc_jobs_ingest import INDEX, EMBED_ID, CHAT_ID, es_client

load_dotenv()
es = es_client()
client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])
MODEL = "mistral-small-latest"

print("Connected to Elastic:", es.ping())
print("Index name:", INDEX)
print("Embedding endpoint:", EMBED_ID)
print("Mistral model:", MODEL)

Connected to Elastic: True
Index name: nyc_jobs
Embedding endpoint: mistral-embeddings
Mistral model: mistral-small-latest


## Q1. Elastic is a database

1. Predict: how many postings are stored?
2. Run the cell below.

In [2]:
count = es.count(index=INDEX)["count"]
print("Postings stored:", count)

resp = es.search(index=INDEX, size=1)
doc = resp["hits"]["hits"][0]["_source"]
print()
print("One document has these keys:")
for k in sorted(doc.keys()):
    print("  -", k)
print()
print("business_title:", doc.get("business_title"))
print("agency:        ", doc.get("agency"))
print("salary range:  ", doc.get("salary_range_from"), "to", doc.get("salary_range_to"), doc.get("salary_frequency"))

Postings stored: 1384

One document has these keys:
  - agency
  - business_title
  - career_level
  - civil_service_title
  - division_work_unit
  - full_time_part_time_indicator
  - job_category
  - job_description
  - job_id
  - level
  - minimum_qual_requirements
  - number_of_positions
  - post_until
  - posting_date
  - posting_type
  - posting_updated
  - process_date
  - residency_requirement
  - salary_frequency
  - salary_range_from
  - salary_range_to
  - search_text
  - title_classification
  - title_code_no
  - work_location
  - work_location_1

business_title: Director, Action Center, Bureau of Harlem Neighborhood Health
agency:         DEPT OF HEALTH/MENTAL HYGIENE
salary range:   98140.0 to 110000.0 Annual


### Check

1. 1384 postings sit in this index.
2. Each posting is one JSON document.
3. Keys are the columns you search.

### Why it matters

1. Everything later searches this one table.

### Flow

1. You run the count cell.
2. Python sends count and search to Elastic.
3. Elastic returns the count and one posting.
4. Python prints them on your screen.
5. Mistral not involved here.

## Q2. Word search finds words

1. Predict: does "python" match a capital "Python"?
2. Run the cell below.

In [3]:
def snippet(text, word, width=150):
    low = (text or "").lower()
    i = low.find(word.lower())
    if i < 0:
        return (text or "")[:width]
    start = max(0, i - width // 2)
    return "..." + text[start:start + width].replace("\n", " ") + "..."

resp = es.search(index=INDEX, size=5, query={"match": {"job_description": "python"}})
print("Total postings with the word python:", resp["hits"]["total"]["value"])
print()
for n, hit in enumerate(resp["hits"]["hits"], 1):
    src = hit["_source"]
    print(f"{n}. {src['business_title']}  ({src['agency']})")
    print("   snippet:", snippet(src.get("job_description"), "python"))
    print()

Total postings with the word python: 15

1. Program Lead, Data Systems & Counts Management  (DEPARTMENT OF TRANSPORTATION)
   snippet: ...ta collection, including performance metrics, quality control, automation, Python-based data workflows and pipelines, and dashboards that make counts ...

2. Strategic Performance Analyst  (DEPARTMENT FOR THE AGING)
   snippet: ...gram platforms and data management using software such as Tableau, SQL, R, Python, etc., to accurately pull data from NYC Aging’s data sources.  •	Bui...

3. GENERATIVE AI DEVELOPER GRADUATE INTERN  (HRA/DEPT OF SOCIAL SERVICES)
   snippet: ...ons utilizing AWS Bedrock and Anthropic Claude models.  •	Review and write Python and/or ReactJS code to support building additional application featu...

4. Data Quality Analyst  (FIRE DEPARTMENT)
   snippet: ...the building of interactive dashboards, primarily using Tableau and modern Python frameworks (Dash/Streamlit). Other core duties include swift fulfill...

5. Senior Policy An

### Check

1. Yes, capital letters do not matter.
2. It only finds the exact word python.
3. "Programming" alone would never match it.

### Why it matters

1. Word search misses synonyms and paraphrases.

### Flow

1. You run the word search cell.
2. Python sends the word python to Elastic.
3. Elastic finds postings containing that exact word.
4. Elastic returns five postings to Python.
5. Python prints titles and snippets onscreen.
6. Mistral not involved here.

## Q3. Meaning search finds meaning

1. Predict: can it find postings without these words?
2. Run the cell below.

In [4]:
phrase = "write code for the city"

sem = es.search(index=INDEX, size=5, query={"semantic": {"field": "search_text", "query": phrase}})
kw  = es.search(index=INDEX, size=5, query={"match": {"job_description": phrase}})

def label(hit):
    src = hit["_source"]
    has_code = "code" in (src.get("job_description") or "").lower()
    return src["business_title"][:40] + ("" if has_code else "  [no 'code']")

sem_titles = [label(h) for h in sem["hits"]["hits"]]
kw_titles  = [label(h) for h in kw["hits"]["hits"]]

print("Search words:", repr(phrase))
print("[no 'code'] = description never says code")
print()
print(f"{'MEANING search (semantic)':<55} | WORD search (match)")
print("-" * 55 + "-+-" + "-" * 50)
for i in range(5):
    a = sem_titles[i] if i < len(sem_titles) else ""
    b = kw_titles[i] if i < len(kw_titles) else ""
    print(f"{a[:55]:<55} | {b[:50]}")

Search words: 'write code for the city'
[no 'code'] = description never says code

MEANING search (semantic)                               | WORD search (match)
--------------------------------------------------------+---------------------------------------------------
Staff Writer  [no 'code']                               | Senior DevOps Engineer, Infrastructure
Assistant Corporation Counsel, Commercia  [no 'code']   | Senior DevOps Engineer, Infrastructure
Project Coordinator, Safety Projects and  [no 'code']   | Senior DevOps Engineer, Infrastructure
Application Developer                                   | UNIX MIDDLEWARE ENGINEER
Data Engineer                                           | Air Permitting Engineering Intern


### Check

1. Yes: three top hits never say code.
2. They are weak: Staff Writer matched write.
3. Word search needs the literal words.

### Why it matters

1. Real apps mix both search types.
2. Mistral rewrites words into job titles.
3. Then search works; tools cover failures.

### Flow

1. You run the side by side cell.
2. Python sends the phrase to Elastic twice.
3. Elastic asks Mistral to embed the phrase.
4. Elastic picks five postings closest in meaning.
5. Elastic also matches literal words, five more.
6. Elastic returns both lists to Python.
7. Python prints both lists side by side.
8. Mistral only made numbers; no chat here.

## Q4. What an embedding is

1. Predict: how many numbers per one sentence?
2. Run the cell below.

In [ ]:
sentences = [
    "software engineer who writes Python",
    "backend developer",
    "registered nurse",
]

try:
    resp = es.inference.inference(inference_id=EMBED_ID, task_type="text_embedding", input=sentences)
except Exception:
    resp = es.perform_request("POST", f"/_inference/text_embedding/{EMBED_ID}",
                              headers={"content-type": "application/json", "accept": "application/json"},
                              body={"input": sentences})
vectors = [item["embedding"] for item in resp["text_embedding"]]

for s, v in zip(sentences, vectors):
    print(f"{s!r}")
    print("   numbers in vector:", len(v))
    print("   first 5 numbers:  ", [round(x, 4) for x in v[:5]])
    print()

def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))

print("Similarity, Python engineer vs backend developer:", round(cosine(vectors[0], vectors[1]), 4))
print("Similarity, Python engineer vs registered nurse: ", round(cosine(vectors[0], vectors[2]), 4))

### Check

1. 1024 numbers stand for one sentence.
2. Engineer and developer score the highest.
3. Nurse scores lower: a different meaning.

### Why it matters

1. Semantic search is just this comparison.

### Flow

1. You run the embedding cell.
2. Python sends three sentences to Elastic.
3. Elastic forwards them to Mistral embeddings.
4. Mistral returns 1024 numbers per sentence.
5. Elastic passes the numbers back to Python.
6. Python computes similarity, prints both scores.

## Q5. Elastic made those embeddings for you

1. Predict: who called Mistral at load, Elastic?
2. Run the cell below.

In [ ]:
mapping = es.indices.get_mapping(index=INDEX)
field = mapping[INDEX]["mappings"]["properties"]["search_text"]
print("search_text field definition:")
print(json.dumps(dict(field), indent=2))
print()

try:
    info = es.inference.get(task_type="text_embedding", inference_id=EMBED_ID)
except Exception:
    info = es.perform_request("GET", f"/_inference/text_embedding/{EMBED_ID}",
                              headers={"accept": "application/json"})
ep = info["endpoints"][0]
settings = ep.get("service_settings", {})
print("Inference endpoint:", ep.get("inference_id"))
print("Task type:         ", ep.get("task_type"))
print("Service:           ", ep.get("service"))
print("Model:             ", settings.get("model"))
print("API key shown?      no, Elastic keeps it hidden")

### Check

1. Elastic called Mistral, not your code.
2. Elastic stores the Mistral key itself.
3. It embeds at load and query time.

### Why it matters

1. You never manage vectors by hand.

### Flow

1. You run the inspection cell.
2. Python asks Elastic for its settings.
3. Elastic shows search_text uses mistral-embeddings.
4. At loading, Elastic sent postings to Mistral.
5. Mistral returned numbers; Elastic stored them.
6. Python prints service and model, no key.

## Q6. Aggregation counts

1. Predict: which agency posts the most jobs?
2. Run the cell below.

In [ ]:
resp = es.search(
    index=INDEX,
    size=0,
    aggs={
        "top_agencies": {"terms": {"field": "agency", "size": 5}},
        "annual": {
            "filter": {"term": {"salary_frequency": "Annual"}},
            "aggs": {"max_pay": {"stats": {"field": "salary_range_to"}}},
        },
    },
)

print("Top 5 agencies by number of postings:")
for n, b in enumerate(resp["aggregations"]["top_agencies"]["buckets"], 1):
    print(f"  {n}. {b['key']}: {b['doc_count']} postings")

s = resp["aggregations"]["annual"]["max_pay"]
print()
print("Top of salary range, annual postings only:")
print("  postings counted:", s["count"])
print("  lowest top pay:  ", round(s["min"] or 0))
print("  average top pay: ", round(s["avg"] or 0))
print("  highest top pay: ", round(s["max"] or 0))

### Check

1. Health Department leads with 233 postings.
2. Size 0 means counts, no documents.
3. Elastic does the math, not Python.

### Why it matters

1. Dashboards and charts come from aggregations.

### Flow

1. You run the counting cell.
2. Python sends the counting request to Elastic.
3. Elastic groups and counts every posting.
4. Elastic returns totals only, no postings.
5. Python prints agencies and pay numbers.
6. Mistral not involved here.

## Q7. Mistral alone

1. Predict: does Mistral know these postings already?
2. Run the cell below.

In [ ]:
resp = client.chat.complete(
    model=MODEL,
    messages=[{"role": "user",
               "content": "How many open NYC city job postings exist right now? Answer in one sentence."}],
)
print("Mistral says:", resp.choices[0].message.content)
print()
print("Elastic actually holds:", es.count(index=INDEX)["count"], "postings")

### Check

1. No, it guessed a number; truth 1384.
2. It cannot see your Elastic index.
3. Its knowledge stops at training time.

### Why it matters

1. Models need your data handed in.

### Flow

1. You run the question cell.
2. Python sends your question to Mistral.
3. Mistral answers from old training memory.
4. Python prints the guess on screen.
5. Python then asks Elastic for truth.
6. Mistral never sees Elastic; it guesses.

## Q8. The bridge, one tool call

1. Predict: who runs the search, Mistral, Python?
2. Run the two cells below, in order.

In [ ]:
tools = [{
    "type": "function",
    "function": {
        "name": "search_jobs",
        "description": "Search NYC city job postings by meaning, with an optional minimum annual salary.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "What kind of job, in plain words."},
                "min_salary": {"type": "number", "description": "Minimum annual salary in dollars."},
            },
            "required": ["query"],
        },
    },
}]

messages = [{"role": "user", "content": "Find city jobs for a Python developer paying over 100k"}]

first = client.chat.complete(model=MODEL, messages=messages, tools=tools, tool_choice="any")
call = first.choices[0].message.tool_calls[0]

print("1 Mistral asked for:")
print("  tool name:", call.function.name)
print("  arguments:", call.function.arguments)

In [ ]:
args = call.function.arguments
args = json.loads(args) if isinstance(args, str) else args

query = {"bool": {"must": [{"semantic": {"field": "search_text", "query": args["query"]}}]}}
if args.get("min_salary"):
    query["bool"]["filter"] = [{"range": {"salary_range_to": {"gte": args["min_salary"]}}}]

hits = es.search(index=INDEX, size=3, query=query)["hits"]["hits"]
results = [
    {"title": h["_source"]["business_title"],
     "agency": h["_source"]["agency"],
     "salary_to": h["_source"].get("salary_range_to")}
    for h in hits
]

print("2 Python ran Elastic, got:")
for r in results:
    print(f"  - {r['title']} ({r['agency']}), up to {r['salary_to']}")
print()

messages.append(first.choices[0].message)
messages.append({"role": "tool", "name": call.function.name,
                 "tool_call_id": call.id, "content": json.dumps(results)})
messages.append({"role": "user", "content": "Answer in one sentence."})
final = client.chat.complete(model=MODEL, messages=messages, tools=tools)

print("3 Mistral answered:")
print(" ", final.choices[0].message.content)

### Check

1. Mistral only wrote the search request.
2. Python ran Elastic and passed results.
3. Retrieve, then generate: the RAG loop.

### Why it matters

1. This loop powers every agent tonight.

### Flow

1. You type the question into Python.
2. Python sends question plus tool to Mistral.
3. Mistral replies: run search_jobs, query, salary.
4. Python sends that search to Elastic.
5. Elastic asks Mistral to embed the query.
6. Elastic returns three postings to Python.
7. Python hands the postings to Mistral.
8. Mistral writes answer; Python prints it.

## Score yourself

1. Q1 right if you said about 1400.
2. Q2 right if you said yes, case-blind.
3. Q3 right if you said yes, meaning wins.
4. Q4 right if you said 1024 numbers.
5. Q5 right if you said Elastic called.
6. Q6 right if you said Health Department.
7. Q7 right if you said it guesses.
8. Q8 right if you said Python runs.

## Next

1. Next: nyc_jobs_matcher.ipynb runs this loop.